In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Introduction](introyt1_tutorial.html) \|\|
[Tensors](tensors_deeper_tutorial.html) \|\|
[Autograd](autogradyt_tutorial.html) \|\| **Building Models** \|\|
[TensorBoard Support](tensorboardyt_tutorial.html) \|\| [Training
Models](trainingyt.html) \|\| [Model Understanding](captumyt.html)

Building Models with PyTorch
============================

Follow along with the video below or on
[youtube](https://www.youtube.com/watch?v=OSqIP-mOWOI).



In [2]:
# Run this cell to load the video
from IPython.display import display, HTML
html_code = """
<div style="margin-top:10px; margin-bottom:10px;">
  <iframe width="560" height="315" src="https://www.youtube.com/embed/OSqIP-mOWOI" frameborder="0" allow="accelerometer; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>
</div>
"""
display(HTML(html_code))



`torch.nn.Module` and `torch.nn.Parameter`
------------------------------------------

In this video, we'll be discussing some of the tools PyTorch makes
available for building deep learning networks.

Except for `Parameter`, the classes we discuss in this video are all
subclasses of `torch.nn.Module`. This is the PyTorch base class meant to
encapsulate behaviors specific to PyTorch Models and their components.

One important behavior of `torch.nn.Module` is registering parameters.
If a particular `Module` subclass has learning weights, these weights
are expressed as instances of `torch.nn.Parameter`. The `Parameter`
class is a subclass of `torch.Tensor`, with the special behavior that
when they are assigned as attributes of a `Module`, they are added to
the list of that modules parameters. These parameters may be accessed
through the `parameters()` method on the `Module` class.

As a simple example, here's a very simple model with two linear layers
and an activation function. We'll create an instance of it and ask it to
report on its parameters:


In [2]:
import torch

# Definimos un modelo muy pequeño heredando de torch.nn.Module
class TinyModel(torch.nn.Module):
    
    def __init__(self):
        # Inicializamos correctamente la clase base
        super(TinyModel, self).__init__()
        
        # Primera capa lineal: recibe 100 características y produce 200 salidas
        self.linear1 = torch.nn.Linear(100, 200)
        # Activación no lineal para introducir capacidad de aprendizaje complejo
        self.activation = torch.nn.ReLU()
        # Segunda capa lineal: reduce de 200 a 10 salidas (por ejemplo, 10 clases)
        self.linear2 = torch.nn.Linear(200, 10)
        # Softmax convierte las salidas en probabilidades normalizadas
        self.softmax = torch.nn.Softmax()
    
    def forward(self, x):
        # Paso 1: transformación lineal inicial
        x = self.linear1(x)
        # Paso 2: activación ReLU
        x = self.activation(x)
        # Paso 3: proyección a espacio de salida
        x = self.linear2(x)
        # Paso 4: normalización tipo probabilidad
        x = self.softmax(x)
        return x

# Creamos una instancia del modelo
tinymodel = TinyModel()

# Mostramos el modelo completo
print('The model:')
print(tinymodel)

# Mostramos solo una subcapa concreta
print('\n\nJust one layer, "linear2":')
print(tinymodel.linear2)

# Recorremos y mostramos todos los parámetros aprendibles del modelo
print('\n\nModel params:')
for param in tinymodel.parameters():
    print(param)

# Recorremos y mostramos solo los parámetros de la capa linear2
print('\n\nLayer "linear2" params:')
for param in tinymodel.linear2.parameters():
    print(param)

The model:
TinyModel(
  (linear1): Linear(in_features=100, out_features=200, bias=True)
  (activation): ReLU()
  (linear2): Linear(in_features=200, out_features=10, bias=True)
  (softmax): Softmax(dim=None)
)


Just one layer, "linear2":
Linear(in_features=200, out_features=10, bias=True)


Model params:
Parameter containing:
tensor([[ 0.0413, -0.0994, -0.0213,  ..., -0.0682, -0.0073, -0.0828],
        [ 0.0808, -0.0814,  0.0211,  ..., -0.0101,  0.0800,  0.0426],
        [-0.0259,  0.0517, -0.0864,  ...,  0.0977,  0.0732, -0.0469],
        ...,
        [ 0.0699,  0.0045,  0.0965,  ..., -0.0479, -0.0013, -0.0687],
        [-0.0802, -0.0234, -0.0529,  ..., -0.0198,  0.0622,  0.0971],
        [-0.0227, -0.0636, -0.0297,  ..., -0.0124,  0.0968, -0.0992]],
       requires_grad=True)
Parameter containing:
tensor([-3.2429e-02,  8.7759e-02,  7.6311e-02,  9.8583e-02,  1.1659e-02,
         7.1353e-02,  2.8573e-02,  1.8480e-02,  3.8061e-02,  7.8072e-02,
        -3.3036e-02, -1.8909e-02,  9.6770e-

This shows the fundamental structure of a PyTorch model: there is an
`__init__()` method that defines the layers and other components of a
model, and a `forward()` method where the computation gets done. Note
that we can print the model, or any of its submodules, to learn about
its structure.

Common Layer Types
==================

Linear Layers
-------------

The most basic type of neural network layer is a *linear* or *fully
connected* layer. This is a layer where every input influences every
output of the layer to a degree specified by the layer's weights. If a
model has *m* inputs and *n* outputs, the weights will be an *m* x *n*
matrix. For example:


In [3]:
# Capa lineal con 3 entradas (simil a 3 caracteristicas de un patrón) y 2 salidas (simil a 2 clases)
lin = torch.nn.Linear(3, 2)
# Se crea un tensor que es un simil a un patrón con 3 características
x = torch.rand(1, 3)
print('Input "x":')
print(x)

# Imprimirá los pesos (6 pesos) y sesgos-bias (2 pesos) de la capa lineal, que son los parámetros aprendibles del modelo
print('\n\nWeight and Bias parameters:')
for param in lin.parameters():
    print(param)

# Imprime la salida de cada una de las 2 neuronas de la capa lineal al recibir el patrón de entrada "x"
y = lin(x)
print('\n\nOutput "Linear(x)":')
print(y)

Input "x":
tensor([[0.9551, 0.1961, 0.7085]])


Weight and Bias parameters:
Parameter containing:
tensor([[ 0.5014,  0.3977, -0.5413],
        [ 0.5276, -0.0225, -0.4234]], requires_grad=True)
Parameter containing:
tensor([-0.0672,  0.3879], requires_grad=True)


Output "Linear(x)":
tensor([[0.1061, 0.5874]], grad_fn=<AddmmBackward0>)


If you do the matrix multiplication of `x` by the linear layer's
weights, and add the biases, you'll find that you get the output vector
`y`.

One other important feature to note: When we checked the weights of our
layer with `lin.weight`, it reported itself as a `Parameter` (which is a
subclass of `Tensor`), and let us know that it's tracking gradients with
autograd. This is a default behavior for `Parameter` that differs from
`Tensor`.

Linear layers are used widely in deep learning models. One of the most
common places you'll see them is in classifier models, which will
usually have one or more linear layers at the end, where the last layer
will have *n* outputs, where *n* is the number of classes the classifier
addresses.

Convolutional Layers
====================

*Convolutional* layers are built to handle data with a high degree of
spatial correlation. They are very commonly used in computer vision,
where they detect close groupings of features which the compose into
higher-level features. They pop up in other contexts too - for example,
in NLP applications, where a word's immediate context (that is, the
other words nearby in the sequence) can affect the meaning of a
sentence.

We saw convolutional layers in action in LeNet5 in an earlier video:


In [4]:
import torch.nn.functional as F


class LeNet(torch.nn.Module):

    def __init__(self):
        # Inicializa la clase base para que PyTorch registre capas y parámetros.
        super(LeNet, self).__init__()
        
        """
        EJEMPLO DE LO QUE SACARIA ESTA CONVOLUCION
        Con una imagen de entrada de 10x10: 1x10x10
        6 "imagenes" de salida (sin padding, stride=1) de 6x6: 6x6x6
        Porque el tamaño espacial queda en:
            10-5+1=6
            
        Se obiene de la formula:
        Tamaño mapas cuadrados = floor( (in + 2padding - dilation(kernel - 1) - 1) / stride + 1 )    
        """
        
        # Primera convolución: 1 canal de entrada (imagen en escala de grises),
        # 6 canales (mapas de características) de salida y kernel 5x5.
        self.conv1 = torch.nn.Conv2d(1, 6, 5)
        # Segunda convolución: recibe 6 canales y produce 16, con kernel 3x3.
        self.conv2 = torch.nn.Conv2d(6, 16, 3)
        # Capa lineal: transforma 16*6*6 características en 120 neuronas.
        self.fc1 = torch.nn.Linear(16 * 6 * 6, 120)  # 6*6 viene del tamaño espacial tras conv+pool
        # Capa lineal intermedia: 120 -> 84.
        self.fc2 = torch.nn.Linear(120, 84)
        # Capa de salida: 84 -> 10 logits (por ejemplo, 10 clases).
        self.fc3 = torch.nn.Linear(84, 10)

    def forward(self, x):
        # Bloque 1: conv1 -> ReLU -> max pooling (2x2).
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        # Bloque 2: conv2 -> ReLU -> max pooling (2x2).
        # Como el kernel de pooling es cuadrado, se puede pasar solo un entero.
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        # Aplana el tensor para pasar de mapas 2D a vector 1D por muestra.
        x = x.view(-1, self.num_flat_features(x))
        # Primera capa densa con activación ReLU.
        x = F.relu(self.fc1(x))
        # Segunda capa densa con activación ReLU.
        x = F.relu(self.fc2(x))
        # Capa final sin activación: devuelve logits.
        x = self.fc3(x)
        return x

    def num_flat_features(self, x):
        # Toma todas las dimensiones excepto la del batch.
        # Así, si x tiene forma (N, C, H, W), el método usa (C, H, W).
        size = x.size()[1:]  # all dimensions except the batch dimension
        num_features = 1
        # Multiplica todas las dimensiones para obtener el total de características.
        # Por ejemplo, para (16, 5, 5), devuelve 16x5x5 = 400.
        for s in size:
            num_features *= s
        return num_features

Let's break down what's happening in the convolutional layers of this
model. Starting with `conv1`:

-   LeNet5 is meant to take in a 1x32x32 black & white image. **The
    first argument to a convolutional layer's constructor is the number
    of input channels.** Here, it is 1. If we were building this model
    to look at 3-color channels, it would be 3.
-   A convolutional layer is like a window that scans over the image,
    looking for a pattern it recognizes. These patterns are called
    *features,* and one of the parameters of a convolutional layer is
    the number of features we would like it to learn. **This is the
    second argument to the constructor is the number of output
    features.** Here, we're asking our layer to learn 6 features.
-   Just above, I likened the convolutional layer to a window - but how
    big is the window? **The third argument is the window or kernel
    size.** Here, the "5" means we've chosen a 5x5 kernel. (If you want
    a kernel with height different from width, you can specify a tuple
    for this argument - e.g., `(3, 5)` to get a 3x5 convolution kernel.)

The output of a convolutional layer is an *activation map* - a spatial
representation of the presence of features in the input tensor. `conv1`
will give us an output tensor of 6x28x28; 6 is the number of features,
and 28 is the height and width of our map. (The 28 comes from the fact
that when scanning a 5-pixel window over a 32-pixel row, there are only
28 valid positions.)

We then pass the output of the convolution through a ReLU activation
function (more on activation functions later), then through a max
pooling layer. The max pooling layer takes features near each other in
the activation map and groups them together. It does this by reducing
the tensor, merging every 2x2 group of cells in the output into a single
cell, and assigning that cell the maximum value of the 4 cells that went
into it. This gives us a lower-resolution version of the activation map,
with dimensions 6x14x14.

Our next convolutional layer, `conv2`, expects 6 input channels
(corresponding to the 6 features sought by the first layer), has 16
output channels, and a 3x3 kernel. It puts out a 16x12x12 activation
map, which is again reduced by a max pooling layer to 16x6x6. Prior to
passing this output to the linear layers, it is reshaped to a 16 \* 6 \*
6 = 576-element vector for consumption by the next layer.

There are convolutional layers for addressing 1D, 2D, and 3D tensors.
There are also many more optional arguments for a conv layer
constructor, including stride length(e.g., only scanning every second or
every third position) in the input, padding (so you can scan out to the
edges of the input), and more. See the
[documentation](https://pytorch.org/docs/stable/nn.html#convolution-layers)
for more information.

Recurrent Layers
================

*Recurrent neural networks* (or *RNNs)* are used for sequential data
-anything from time-series measurements from a scientific instrument to
natural language sentences to DNA nucleotides. An RNN does this by
maintaining a *hidden state* that acts as a sort of memory for what it
has seen in the sequence so far.

The internal structure of an RNN layer - or its variants, the LSTM (long
short-term memory) and GRU (gated recurrent unit) - is moderately
complex and beyond the scope of this video, but we'll show you what one
looks like in action with an LSTM-based part-of-speech tagger (a type of
classifier that tells you if a word is a noun, verb, etc.):


In [5]:
class LSTMTagger(torch.nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        # Inicializa la clase base para registrar submódulos y parámetros.
        super(LSTMTagger, self).__init__()
        # Guarda el tamaño del estado oculto de la LSTM.
        self.hidden_dim = hidden_dim

        # Capa de embeddings: convierte índices de palabras en vectores densos.
        self.word_embeddings = torch.nn.Embedding(vocab_size, embedding_dim)

        # La LSTM recibe embeddings y devuelve estados ocultos de tamaño hidden_dim.
        self.lstm = torch.nn.LSTM(embedding_dim, hidden_dim)

        # Capa lineal que proyecta del espacio oculto al espacio de etiquetas.
        self.hidden2tag = torch.nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        # 1) Convierte cada índice de palabra en su embedding.
        embeds = self.word_embeddings(sentence)
        # 2) Ejecuta la LSTM. Se reacomoda a (longitud_secuencia, batch=1, embedding_dim).
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        # 3) Proyecta cada salida temporal al espacio de etiquetas.
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        # 4) Convierte logits a log-probabilidades por etiqueta.
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores

The constructor has four arguments:

-   `vocab_size` is the number of words in the input vocabulary. Each
    word is a one-hot vector (or unit vector) in a
    `vocab_size`-dimensional space.
-   `tagset_size` is the number of tags in the output set.
-   `embedding_dim` is the size of the *embedding* space for the
    vocabulary. An embedding maps a vocabulary onto a low-dimensional
    space, where words with similar meanings are close together in the
    space.
-   `hidden_dim` is the size of the LSTM's memory.

The input will be a sentence with the words represented as indices of
one-hot vectors. The embedding layer will then map these down to an
`embedding_dim`-dimensional space. The LSTM takes this sequence of
embeddings and iterates over it, fielding an output vector of length
`hidden_dim`. The final linear layer acts as a classifier; applying
`log_softmax()` to the output of the final layer converts the output
into a normalized set of estimated probabilities that a given word maps
to a given tag.

If you'd like to see this network in action, check out the [Sequence
Models and LSTM
Networks](https://pytorch.org/tutorials/beginner/nlp/sequence_models_tutorial.html)
tutorial on pytorch.org.

Transformers
============

*Transformers* are multi-purpose networks that have taken over the state
of the art in NLP with models like BERT. A discussion of transformer
architecture is beyond the scope of this video, but PyTorch has a
`Transformer` class that allows you to define the overall parameters of
a transformer model - the number of attention heads, the number of
encoder & decoder layers, dropout and activation functions, etc. (You
can even build the BERT model from this single class, with the right
parameters!) The `torch.nn.Transformer` class also has classes to
encapsulate the individual components (`TransformerEncoder`,
`TransformerDecoder`) and subcomponents (`TransformerEncoderLayer`,
`TransformerDecoderLayer`). For details, check out the
[documentation](https://pytorch.org/docs/stable/nn.html#transformer-layers)
on transformer classes.

Other Layers and Functions
--------------------------

Data Manipulation Layers
========================

There are other layer types that perform important functions in models,
but don't participate in the learning process themselves.

**Max pooling** (and its twin, min pooling) reduce a tensor by combining
cells, and assigning the maximum value of the input cells to the output
cell (we saw this). For example:


In [ ]:
# Creamos un tensor aleatorio con forma (batch=1, alto=6, ancho=6).
my_tensor = torch.rand(1, 6, 6)
print(f'Tensor original: {my_tensor.shape}\n{my_tensor}')

# Definimos max pooling 2D con ventana 3x3.
# Por defecto, el stride será 3, así que reduce la resolución espacial.
# Sino se indica nada, el stride es igual al tamaño del kernel, así que en este caso también sería 3.
maxpool_layer = torch.nn.MaxPool2d(3)

# Aplicamos max pooling: en cada bloque 3x3 toma el valor máximo.
# Para una entrada 6x6 y kernel 3, la salida esperada es 2x2 (manteniendo batch=1).
print(f'Tensor después de max pooling2D de 3x3: {maxpool_layer(my_tensor).shape}\n{maxpool_layer(my_tensor)}')

Tensor original: torch.Size([1, 6, 6])
tensor([[[0.8728, 0.7173, 0.8110, 0.7741, 0.9149, 0.0347],
         [0.2961, 0.8114, 0.7794, 0.8814, 0.0957, 0.2769],
         [0.5242, 0.2726, 0.5837, 0.7474, 0.7540, 0.3297],
         [0.3730, 0.8931, 0.0339, 0.9791, 0.6268, 0.2300],
         [0.0324, 0.0654, 0.4662, 0.4852, 0.1951, 0.4112],
         [0.5942, 0.4094, 0.1493, 0.5029, 0.5677, 0.3987]]])
Tensor después de max pooling2D de 3x3: torch.Size([1, 2, 2])
tensor([[[0.8728, 0.9149],
         [0.8931, 0.9791]]])


If you look closely at the values above, you'll see that each of the
values in the maxpooled output is the maximum value of each quadrant of
the 6x6 input.

**Normalization layers** re-center and normalize the output of one layer
before feeding it to another. Centering and scaling the intermediate
tensors has a number of beneficial effects, such as letting you use
higher learning rates without exploding/vanishing gradients.


In [6]:
# Creamos un tensor aleatorio de forma (batch=1,  longitud=4, features=4)
# luego lo escalamos y desplazamos para que sus valores sean más grandes.
my_tensor = torch.rand(1, 4, 4) * 20 + 5
print(f'my_tensor: {my_tensor.shape}\n{my_tensor}')

# Mostramos la media global del tensor original.
print(f'Media global del tensor original: {my_tensor.mean()}')

# Definimos una capa BatchNorm1d para 4 características/canales.
"""
En BatchNorm1d, para cada característica se toma cada valor de entrada x y primero se normaliza así:

x_norm = (x - media_batch) / sqrt(var_batch + eps)

Aquí, media_batch y var_batch son la media y varianza del batch (calculadas por característica), y eps es un valor pequeño para evitar inestabilidad numérica.
Después, BatchNorm1d aplica una transformación aprendible:

y = gamma * x_norm + beta

Por eso, el valor final y no tiene por qué ser igual a x_norm. Solo serían iguales si gamma = 1 y beta = 0. Esta segunda parte permite que la red aprenda cuánto escalar (gamma) y desplazar (beta) la señal normalizada para mejorar el entrenamiento.
"""
norm_layer = torch.nn.BatchNorm1d(4)

# Aplicamos normalización por batch: re-centra y re-escala activaciones.
normed_tensor = norm_layer(my_tensor)
print(f'Tensor normalizado: {normed_tensor.shape}\n{normed_tensor}')

# Mostramos la media global tras normalizar (debería acercarse a 0).
print(f'Media global del tensor normalizado: {normed_tensor.mean()}')

my_tensor: torch.Size([1, 4, 4])
tensor([[[11.4069,  6.4118,  8.1026, 17.2891],
         [21.3012, 19.0928, 14.2213, 11.7268],
         [13.6245, 20.7198,  9.9554, 22.8771],
         [ 5.4782, 13.3484,  9.1726, 14.9378]]])
Media global del tensor original: 13.729145050048828
Tensor normalizado: torch.Size([1, 4, 4])
tensor([[[ 0.1455, -1.0571, -0.6500,  1.5617],
         [ 1.2414,  0.6600, -0.6224, -1.2790],
         [-0.6066,  0.7512, -1.3087,  1.1641],
         [-1.4230,  0.7078, -0.4228,  1.1381]]],
       grad_fn=<NativeBatchNormBackward0>)
Media global del tensor normalizado: 7.450580596923828e-09


Running the cell above, we've added a large scaling factor and offset to
an input tensor; you should see the input tensor's `mean()` somewhere in
the neighborhood of 15. After running it through the normalization
layer, you can see that the values are smaller, and grouped around zero
- in fact, the mean should be very small (\> 1e-8).

This is beneficial because many activation functions (discussed below)
have their strongest gradients near 0, but sometimes suffer from
vanishing or exploding gradients for inputs that drive them far away
from zero. Keeping the data centered around the area of steepest
gradient will tend to mean faster, better learning and higher feasible
learning rates.

**Dropout layers** are a tool for encouraging *sparse representations*
in your model - that is, pushing it to do inference with less data.

Dropout layers work by randomly setting parts of the input tensor
*during training* - dropout layers are always turned off for inference.
This forces the model to learn against this masked or reduced dataset.
For example:


In [10]:
# Tensor de ejemplo con forma (batch=1, 4x4).
my_tensor = torch.rand(1, 4, 4)
print(f'my_tensor: {my_tensor.shape}\n{my_tensor}')

# Dropout con probabilidad p=0.4: en entrenamiento anula aleatoriamente
# aproximadamente el 40% de los elementos en cada pasada.
dropout = torch.nn.Dropout(p=0.4)

# Primera aplicación de dropout sobre el mismo tensor de entrada.
# OJO, "my_tensor" no se modifica inplace, sino que se devuelve un nuevo tensor con algunos valores anulados. Por tanto si igualamos la función dropout(my_tensor) a una variable, esa variable contendría el tensor con algunos valores anulados, pero "my_tensor" seguiría intacto.
print(f'\nDropout1 aplicado a my_tensor:\n{dropout(my_tensor)}')
# Segunda aplicación: el patrón de ceros cambia porque la máscara es aleatoria.
print(f'\nDropout2 aplicado a my_tensor:\n{dropout(my_tensor)}')

my_tensor: torch.Size([1, 4, 4])
tensor([[[0.6872, 0.5347, 0.5603, 0.5636],
         [0.1668, 0.8919, 0.5922, 0.3237],
         [0.3476, 0.2967, 0.5836, 0.1784],
         [0.5198, 0.9086, 0.1253, 0.5512]]])

Dropout1 aplicado a my_tensor:
tensor([[[1.1453, 0.0000, 0.9339, 0.9394],
         [0.2780, 1.4865, 0.9869, 0.5396],
         [0.5794, 0.0000, 0.9727, 0.2974],
         [0.0000, 1.5143, 0.2089, 0.0000]]])

Dropout2 aplicado a my_tensor:
tensor([[[1.1453, 0.0000, 0.9339, 0.9394],
         [0.2780, 0.0000, 0.0000, 0.5396],
         [0.0000, 0.0000, 0.0000, 0.2974],
         [0.0000, 0.0000, 0.2089, 0.0000]]])


Above, you can see the effect of dropout on a sample tensor. You can use
the optional `p` argument to set the probability of an individual weight
dropping out; if you don't it defaults to 0.5.

Activation Functions
====================

Activation functions make deep learning possible. A neural network is
really a program - with many parameters - that *simulates a mathematical
function*. If all we did was multiple tensors by layer weights
repeatedly, we could only simulate *linear functions;* further, there
would be no point to having many layers, as the whole network would
reduce could be reduced to a single matrix multiplication. Inserting
*non-linear* activation functions between layers is what allows a deep
learning model to simulate any function, rather than just linear ones.

`torch.nn.Module` has objects encapsulating all of the major activation
functions including ReLU and its many variants, Tanh, Hardtanh, sigmoid,
and more. It also includes other functions, such as Softmax, that are
most useful at the output stage of a model.

Loss Functions
==============

Loss functions tell us how far a model's prediction is from the correct
answer. PyTorch contains a variety of loss functions, including common
MSE (mean squared error = L2 norm), Cross Entropy Loss and Negative
Likelihood Loss (useful for classifiers), and others.
